In [2]:
import sys
print(sys.executable)
print(sys.version)
%pip install -r requirements.txt

c:\Users\leo.langou\AppData\Local\miniconda3\envs\space-logistics\python.exe
3.11.16 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:36:16) [MSC v.1942 64 bit (AMD64)]
  Using cached sgp4-2.27-cp311-abi3-win_amd64.whl.metadata (35 kB)
  Using cached spacetrack-2.1.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached kmapper-2.1.0-py3-none-any.whl.metadata (4.9 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached idna-3.19-py3-none-any.whl.metadata (9.2 kB)
  Using cached anyio-4.15.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached filelock-3.32.5-py3-none-any.whl.metadata (2.0 kB)
  Using cached httpx2-2.12.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached represent-2.2.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached rush-2021.4.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached cycler-0.12.1-py3-no

In [3]:
from us_geo_timeseries import (
    get_current_us_geo_catalog,
    download_gp_history,
    build_uniform_timeseries
)

import pandas as pd

In [5]:
start = pd.Timestamp("2026-08-01T00:00:00Z")
end = pd.Timestamp("2026-09-01T00:00:00Z")

cadence = "6h"

catalog = get_current_us_geo_catalog()

print(f"US GEO satellites found: {len(catalog)}")

display(
    catalog[
        [
            "NORAD_CAT_ID",
            "OBJECT_NAME",
            "OWNER",
            "PERIOD"
        ]
    ].head(30)
)

US GEO satellites found: 124


,NORAD_CAT_ID,OBJECT_NAME,OWNER,PERIOD
0,19548,TDRS 3,US,1436.00
1,20253,FLTSATCOM 8 (USA 46),US,1435.99
2,21639,TDRS 5,US,1436.05
3,22314,TDRS 6,US,1435.99
4,22787,UFO 2 (USA 95),US,1436.07
5,23467,UFO 4 (USA 108),US,1436.04
6,23613,TDRS 7,US,1436.03
7,23712,USA 115 (MILSTAR-1 2),US,1436.09
8,25019,USA 134,US,1436.11
9,25336,USA 139,US,1436.13


In [ ]:
ids = catalog["NORAD_CAT_ID"].tolist()

print(f"Requesting history for {len(ids)} satellites")

records = download_gp_history(
    ids=ids,
    start=start,
    end=end
)

print(f"Historical element sets downloaded: {len(records):,}")

Requesting history for 124 satellites
Historical element sets downloaded: 12,601


In [9]:
records[0]

{'CCSDS_OMM_VERS': '3.0',
 'COMMENT': 'GENERATED VIA SPACE-TRACK.ORG API',
 'CREATION_DATE': '2026-07-25T18:12:52',
 'ORIGINATOR': '18 SPCS',
 'OBJECT_NAME': 'TDRS 3',
 'OBJECT_ID': '1988-091B',
 'CENTER_NAME': 'EARTH',
 'REF_FRAME': 'TEME',
 'TIME_SYSTEM': 'UTC',
 'MEAN_ELEMENT_THEORY': 'SGP4',
 'EPOCH': '2026-07-25T05:48:44.275968',
 'MEAN_MOTION': '1.00278149',
 'ECCENTRICITY': '0.00375323',
 'INCLINATION': '12.5726',
 'RA_OF_ASC_NODE': '340.7465',
 'ARG_OF_PERICENTER': '355.3115',
 'MEAN_ANOMALY': '5.1431',
 'EPHEMERIS_TYPE': '0',
 'CLASSIFICATION_TYPE': 'U',
 'NORAD_CAT_ID': '19548',
 'ELEMENT_SET_NO': '999',
 'REV_AT_EPOCH': '12576',
 'BSTAR': '0.00000000000000',
 'MEAN_MOTION_DOT': '-0.00000295',
 'MEAN_MOTION_DDOT': '0.0000000000000',
 'SEMIMAJOR_AXIS': '42162.948',
 'PERIOD': '1436.006',
 'APOAPSIS': '35943.060',
 'PERIAPSIS': '35626.566',
 'OBJECT_TYPE': 'PAYLOAD',
 'RCS_SIZE': 'LARGE',
 'COUNTRY_CODE': 'US',
 'LAUNCH_DATE': '1988-09-29',
 'SITE': 'AFETR',
 'DECAY_DATE': None

In [10]:
history_df = pd.DataFrame(records)

history_df["NORAD_CAT_ID"] = pd.to_numeric(
    history_df["NORAD_CAT_ID"]
)

history_df["EPOCH"] = pd.to_datetime(
    history_df["EPOCH"],
    utc=True
)

counts = (
    history_df
    .groupby(["NORAD_CAT_ID", "OBJECT_NAME"])
    .size()
    .reset_index(name="element_sets")
    .sort_values("element_sets", ascending=False)
)

display(counts)

,NORAD_CAT_ID,OBJECT_NAME,element_sets
34,35491,EWS-G3,159
53,39070,TDRS 11,159
79,45465,AEHF 6 (USA 298),158
86,51850,GOES 18,157
18,27954,HORIZONS 1 (GALAXY 13),157
...,...,...,...
92,56372,GS-1,32
78,44625,MEV-1,19
75,43339,USA 283,13
89,55263,USA 342,13


In [11]:
ts = build_uniform_timeseries(
    catalog=catalog,
    records=records,
    start=start,
    end=end,
    cadence=cadence
)

print(ts.shape)

display(ts.head(20))

(12732, 13)


,time_utc,norad_cat_id,object_name,element_epoch_utc,element_age_hours,sgp4_error,sgp4_error_text,x_teme_km,y_teme_km,z_teme_km,vx_teme_km_s,vy_teme_km_s,vz_teme_km_s
0,2026-08-01 00:00:00+00:00,19548,TDRS 3,2026-08-01 08:02:58.602336+00:00,8.049612,0,,-6725.273454,-40587.857974,-9038.550810,3.031959,-0.516333,0.114116
1,2026-08-01 00:00:00+00:00,20253,FLTSATCOM 8 (USA 46),2026-07-31 18:19:43.953024+00:00,5.671124,0,,10468.889020,39825.632594,9017.061486,-2.977312,0.767888,0.063771
2,2026-08-01 00:00:00+00:00,21639,TDRS 5,2026-08-01 02:41:38.617728+00:00,2.694060,0,,-33360.053769,25234.363924,5333.065603,-1.876702,-2.349930,-0.638423
3,2026-08-01 00:00:00+00:00,22314,TDRS 6,2026-07-31 20:31:58.854720+00:00,3.466985,0,,-4388.865482,-40639.747274,-10318.267287,3.057834,-0.323439,-0.037971
4,2026-08-01 00:00:00+00:00,22787,UFO 2 (USA 95),2026-08-01 02:25:16.476096+00:00,2.421243,0,,39145.775595,-15218.973962,-3689.285363,1.141344,2.796788,0.574858
5,2026-08-01 00:00:00+00:00,23467,UFO 4 (USA 108),2026-08-01 05:29:38.586048+00:00,5.494052,0,,-22121.301651,35249.341569,6697.869294,-2.615513,-1.609688,-0.166627
6,2026-08-01 00:00:00+00:00,23613,TDRS 7,2026-07-31 19:52:04.842912+00:00,4.131988,0,,32684.390746,25487.346805,7543.193067,-1.937589,2.348246,0.449143
7,2026-08-01 00:00:00+00:00,23712,USA 115 (MILSTAR-1 2),2026-08-01 03:46:38.781984+00:00,3.777439,0,,-32467.285012,-26202.351807,-6074.231892,1.959459,-2.291534,-0.603661
8,2026-08-01 00:00:00+00:00,25019,USA 134,2026-08-01 07:04:27.568992+00:00,7.074325,0,,-40208.627603,-12691.880916,656.578035,0.892363,-2.861948,-0.681374
9,2026-08-01 00:00:00+00:00,25967,UFO 10 (USA 146),2026-07-31 19:19:57.478368+00:00,4.667367,0,,11835.219719,-39843.745751,-7102.937844,2.941312,0.889996,-0.094821


In [12]:
good = ts[
    ts["sgp4_error"] == 0
].copy()

print(f"Valid rows: {len(good):,}")
print(f"Satellites: {good['norad_cat_id'].nunique()}")
print(f"Time steps: {good['time_utc'].nunique()}")

Valid rows: 12,732
Satellites: 102
Time steps: 125


In [13]:
ts["sgp4_error"].value_counts(dropna=False)

sgp4_error
0    12732
Name: count, dtype: int64

In [14]:
good["element_age_hours"].describe()

count    12732.000000
mean         5.012946
std          8.974749
min          0.000218
25%          1.166019
50%          2.867854
75%          5.346242
max        166.663152
Name: element_age_hours, dtype: float64

In [15]:
t = good["time_utc"].min()

cloud_df = good.loc[
    good["time_utc"] == t,
    [
        "norad_cat_id",
        "object_name",
        "x_teme_km",
        "y_teme_km",
        "z_teme_km"
    ]
]

print(f"Timestamp: {t}")
print(f"Satellites in point cloud: {len(cloud_df)}")

display(cloud_df)

Timestamp: 2026-08-01 00:00:00+00:00
Satellites in point cloud: 101


,norad_cat_id,object_name,x_teme_km,y_teme_km,z_teme_km
0,19548,TDRS 3,-6725.273454,-40587.857974,-9038.550810
1,20253,FLTSATCOM 8 (USA 46),10468.889020,39825.632594,9017.061486
2,21639,TDRS 5,-33360.053769,25234.363924,5333.065603
3,22314,TDRS 6,-4388.865482,-40639.747274,-10318.267287
4,22787,UFO 2 (USA 95),39145.775595,-15218.973962,-3689.285363
...,...,...,...,...,...
96,65160,NTS-3,-40129.229823,-12401.615689,2002.725401
97,66454,VIASAT-3 F2,-26733.619007,-32602.371030,1.926855
98,68126,ECHOSTAR 25,-39705.538755,-14176.243479,25.897167
99,68893,VIASAT-3 F3,-13162.536062,40055.339171,-3.777779


In [16]:
cloud = cloud_df[
    ["x_teme_km", "y_teme_km", "z_teme_km"]
].to_numpy()

print(cloud.shape)
print(cloud[:5])

(101, 3)
[[ -6725.27345443 -40587.85797397  -9038.55080976]
 [ 10468.88902015  39825.63259406   9017.06148581]
 [-33360.05376894  25234.36392403   5333.06560254]
 [ -4388.86548179 -40639.74727393 -10318.26728712]
 [ 39145.77559451 -15218.97396244  -3689.28536291]]


In [18]:
history_ids = set(
    pd.to_numeric(
        history_df["NORAD_CAT_ID"],
        errors="coerce"
    ).dropna().astype(int)
)

catalog_ids = set(catalog["NORAD_CAT_ID"])

missing_ids = catalog_ids - history_ids

missing_catalog = catalog[
    catalog["NORAD_CAT_ID"].isin(missing_ids)
][
    [
        "NORAD_CAT_ID",
        "OBJECT_NAME",
        "OWNER",
        "PERIOD"
    ]
].sort_values("NORAD_CAT_ID")

print("Current catalog satellites:", len(catalog_ids))
print("Satellites with GP_HISTORY:", len(history_ids))
print("Missing from GP_HISTORY:", len(missing_ids))

display(missing_catalog)

Current catalog satellites: 124
Satellites with GP_HISTORY: 102
Missing from GP_HISTORY: 22


,NORAD_CAT_ID,OBJECT_NAME,OWNER,PERIOD
9,25336,USA 139,US,1436.13
19,27937,USA 171,US,1436.09
35,33490,USA 202,US,1436.03
40,35815,USA 207,US,1436.10
50,37232,USA 223,US,1436.11
56,38466,USA 236,US,1436.13
57,38528,USA 237,US,1436.07
66,39652,USA 250,US,1436.10
67,39751,USA 252,US,1436.10
68,40208,USA 257 (CLIO),US,1435.99


In [19]:
coverage = (
    good.groupby("time_utc")
    ["norad_cat_id"]
    .nunique()
)

display(coverage.describe())

print("\nMinimum satellites:", coverage.min())
print("Maximum satellites:", coverage.max())

display(
    coverage[
        coverage < coverage.max()
    ].to_frame("satellite_count")
)

count    125.000000
mean     101.856000
std        0.352503
min      101.000000
25%      102.000000
50%      102.000000
75%      102.000000
max      102.000000
Name: norad_cat_id, dtype: float64


Minimum satellites: 101
Maximum satellites: 102


,satellite_count
time_utc,
2026-08-01 00:00:00+00:00,101
2026-08-01 06:00:00+00:00,101
2026-08-01 12:00:00+00:00,101
2026-08-01 18:00:00+00:00,101
2026-08-02 00:00:00+00:00,101
2026-08-02 06:00:00+00:00,101
2026-08-02 12:00:00+00:00,101
2026-08-02 18:00:00+00:00,101
2026-08-03 00:00:00+00:00,101


In [20]:
all_good_ids = set(good["norad_cat_id"].unique())

t0 = good["time_utc"].min()

t0_ids = set(
    good.loc[
        good["time_utc"] == t0,
        "norad_cat_id"
    ]
)

missing_at_t0 = all_good_ids - t0_ids

display(
    catalog[
        catalog["NORAD_CAT_ID"].isin(missing_at_t0)
    ][
        ["NORAD_CAT_ID", "OBJECT_NAME", "PERIOD"]
    ]
)

,NORAD_CAT_ID,OBJECT_NAME,PERIOD
90,44625,MEV-1,1436.11


In [21]:
import numpy as np

good["radius_km"] = np.sqrt(
    good["x_teme_km"]**2 +
    good["y_teme_km"]**2 +
    good["z_teme_km"]**2
)

good["radius_km"].describe()

count    12732.000000
mean     42163.634710
std         31.452928
min      41966.750846
25%      42159.266099
50%      42164.280371
75%      42169.041944
max      42342.843656
Name: radius_km, dtype: float64

In [22]:
display(
    good[
        [
            "norad_cat_id",
            "object_name",
            "time_utc",
            "radius_km"
        ]
    ]
    .sort_values("radius_km")
    .head(20)
)

,norad_cat_id,object_name,time_utc,radius_km
835,32708,AMC-14,2026-08-03 00:00:00+00:00,41966.750846
431,32708,AMC-14,2026-08-02 00:00:00+00:00,41967.187698
1239,32708,AMC-14,2026-08-04 00:00:00+00:00,41967.738845
1643,32708,AMC-14,2026-08-05 00:00:00+00:00,41969.403891
27,32708,AMC-14,2026-08-01 00:00:00+00:00,41970.036769
2049,32708,AMC-14,2026-08-06 00:00:00+00:00,41970.507048
2457,32708,AMC-14,2026-08-07 00:00:00+00:00,41972.054700
2865,32708,AMC-14,2026-08-08 00:00:00+00:00,41974.615691
3273,32708,AMC-14,2026-08-09 00:00:00+00:00,41976.150991
3681,32708,AMC-14,2026-08-10 00:00:00+00:00,41978.110422
